In [158]:
import os
import pandas as pd
import numpy as np

In [159]:
#load the dataset
df = pd.read_csv('./data/imdb_master.csv', encoding='latin-1')
df = df[['review', 'label']]

In [160]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [161]:
# look at dataset
print(df.head())

                                              review label
0  Once again Mr. Costner has dragged out a movie...   neg
1  This is an example of why the majority of acti...   neg
2  First of all I hate those moronic rappers, who...   neg
3  Not even the Beatles could write songs everyon...   neg
4  Brass pictures (movies is not a fitting word f...   neg


In [162]:
df = df[df["label"].isin(["neg", "pos"])].copy()

df["label_num"] = df["label"].map({
    "neg": 0,
    "pos": 1
})

In [163]:
df.describe()

,label_num
count,50000.000000
mean,0.500000
std,0.500005
min,0.000000
25%,0.000000
50%,0.500000
75%,1.000000
max,1.000000


In [164]:
#drop na values
df = df.dropna()

In [165]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   label      50000 non-null  str  
 2   label_num  50000 non-null  int64
dtypes: int64(1), str(2)
memory usage: 1.1 MB


In [166]:
#create a new column 'label_num' to convert the labels to numerical values
df['label_num'] = df['label'].map({'neg': 0, 'pos': 1})

In [167]:
df['label_num'].value_counts()

label_num
0    25000
1    25000
Name: count, dtype: int64

In [168]:
df.head()

,review,label,label_num
0,Once again Mr. Costner has dragged out a movie...,neg,0
1,This is an example of why the majority of acti...,neg,0
2,"First of all I hate those moronic rappers, who...",neg,0
3,Not even the Beatles could write songs everyon...,neg,0
4,Brass pictures (movies is not a fitting word f...,neg,0


In [169]:
print(df.shape)
print(df.head())
print(df.columns)
print(df.info())
print(df['label'].value_counts())

(50000, 3)
                                              review label  label_num
0  Once again Mr. Costner has dragged out a movie...   neg          0
1  This is an example of why the majority of acti...   neg          0
2  First of all I hate those moronic rappers, who...   neg          0
3  Not even the Beatles could write songs everyon...   neg          0
4  Brass pictures (movies is not a fitting word f...   neg          0
Index(['review', 'label', 'label_num'], dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   label      50000 non-null  str  
 2   label_num  50000 non-null  int64
dtypes: int64(1), str(2)
memory usage: 1.1 MB
None
label
neg    25000
pos    25000
Name: count, dtype: int64


In [170]:
print(df.shape)

(50000, 3)


In [171]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label_num"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label_num"]
)

In [172]:
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (40000, 3)
Validation: (5000, 3)
Test: (5000, 3)


In [173]:
import re

def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    return text.split()

In [174]:
text = "This is a sample text with HTML tags <b>bold</b> and special characters! @#"

print("Original text:", text)
tokenized_text = tokenize(text)
print("Tokenized text:", tokenized_text)

Original text: This is a sample text with HTML tags <b>bold</b> and special characters! @#
Tokenized text: ['this', 'is', 'a', 'sample', 'text', 'with', 'html', 'tags', 'bboldb', 'and', 'special', 'characters']


In [175]:
from collections import Counter

counter = Counter()

for review in train_df["review"]:
    tokens = tokenize(review)
    counter.update(tokens)

In [176]:
vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word, frequency in counter.items():
    vocab[word] = len(vocab)

In [177]:
print("Vocabulary size:", len(vocab))
print(list(vocab.items())[:20])

Vocabulary size: 159144
[('<PAD>', 0), ('<UNK>', 1), ('although', 2), ('this', 3), ('movie', 4), ('has', 5), ('some', 6), ('weaknesses', 7), ('it', 8), ('is', 9), ('worth', 10), ('seeing', 11), ('i', 12), ('chose', 13), ('because', 14), ('of', 15), ('the', 16), ('cast', 17), ('and', 18), ('applaud', 19)]


In [178]:
def encode(text, vocab):
    tokens = tokenize(text)

    return [
        vocab.get(token, vocab["<UNK>"])
        for token in tokens
    ]

In [179]:
review = "This product is amazing"

encoded = encode(review, vocab)

print(encoded)

[3, 7204, 9, 1341]


In [180]:
MAX_LEN = 200

def pad_sequence(sequence, max_len, pad_id=0):

    if len(sequence) < max_len:
        sequence = sequence + [pad_id] * (max_len - len(sequence))

    else:
        sequence = sequence[:max_len]

    return sequence

In [181]:
sequence = [5, 5091, 7, 1427]

padded = pad_sequence(sequence, MAX_LEN)

print(len(padded))
print(padded[:10])

200
[5, 5091, 7, 1427, 0, 0, 0, 0, 0, 0]


In [182]:
import torch
from torch.utils.data import Dataset

In [183]:
class ReviewDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        sequence = self.sequences[idx]
        label = self.labels[idx]

        length = (sequence != 0).sum().item()

        return sequence, length, label

In [193]:

train_dataset = ReviewDataset(
    train_df,
    vocab,
)

val_dataset = ReviewDataset(
    val_df,
    vocab,
)

test_dataset = ReviewDataset(
    test_df,
    vocab,
)

In [195]:
print(type(train_dataset))
print(train_dataset.columns)
print(train_dataset.head())

<class '__main__.ReviewDataset'>


AttributeError: 'ReviewDataset' object has no attribute 'columns'

In [194]:
x, y = train_dataset[0]

print("Input:", x)
print("Input shape:", x.shape)
print("Label:", y)

KeyError: 0

In [187]:
from torch.utils.data import DataLoader


BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
x, y = next(iter(train_loader))

print("Input shape:", x.shape)
print("Label shape:", y.shape)

Input shape: torch.Size([32, 200])
Label shape: torch.Size([32])


In [ ]:
from torch.nn.utils.rnn import pack_padded_sequence
from torch import nn

class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x, lengths):

        embedded = self.embedding(x)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        output, hidden = self.rnn(packed)

        last_hidden = hidden[-1]

        logits = self.fc(last_hidden)

        return logits.squeeze(1)

In [ ]:
VOCAB_SIZE = len(vocab)
EMBEDDING_DIM = 128
HIDDEN_DIM = 128

model = SentimentRNN(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM
)

print(model)

SentimentRNN(
  (embedding): Embedding(159144, 128, padding_idx=0)
  (rnn): RNN(128, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)


In [ ]:
model.to(device)

SentimentRNN(
  (embedding): Embedding(159144, 128, padding_idx=0)
  (rnn): RNN(128, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)

In [ ]:
# import optimizer and loss function
import torch.optim as optim

# BCE
criterion = nn.BCEWithLogitsLoss()



In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
x, y = next(iter(train_loader))

x = x.to(device)
y = y.to(device)

optimizer.zero_grad()

logits = model(x)

loss = criterion(logits, y)

loss.backward()

optimizer.step()

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):

    model.train()

    total_loss = 0

    for x, lengths, y in train_loader:

        x = x.to(device)
        y = y.float().to(device)
        lengths = lengths.to(device)

        optimizer.zero_grad()

        logits = model(x, lengths)

        loss = criterion(logits, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
loss = train_one_epoch(
    model,
    train_loader,
    criterion,
    optimizer
)

print("Training Loss:", loss)

Training Loss: 0.6962986080646515


In [ ]:
def evaluate(model, loader, criterion):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for x,lengths, y in loader:
            x = x.to(device)
            y = y.to(device)
            lengths = lengths.to(device)
            logits = model(x, lengths)

            loss = criterion(logits, y)

            total_loss += loss.item()

            predictions = (torch.sigmoid(logits) >= 0.5).float()

            correct += (predictions == y).sum().item()
            total += y.size(0)

    average_loss = total_loss / len(loader)
    accuracy = correct / total

    return average_loss, accuracy

In [ ]:
EPOCHS = 5

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer
    )

    val_loss, val_accuracy = evaluate(
        model,
        val_loader,
        criterion
    )

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.4f}"
    )

KeyboardInterrupt: 

In [ ]:
old_weight = model.rnn.weight_ih_l0[0, 0].item()

model.train()
x, y = next(iter(train_loader))
x = x.to(device)
y = y.to(device)
optimizer.zero_grad()

logits = model(x)
loss = criterion(logits, y)

loss.backward()
optimizer.step()

new_weight = model.rnn.weight_ih_l0[0, 0].item()

print("Old:", old_weight)
print("New:", new_weight)
print("Changed:", old_weight != new_weight)

Old: -0.23021312057971954
New: -0.23021641373634338
Changed: True


In [ ]:
train_loss, train_acc = evaluate(
    model,
    train_loader,
    criterion
)

val_loss, val_acc = evaluate(
    model,
    val_loader,
    criterion
)

print("Train accuracy:", train_acc)
print("Validation accuracy:", val_acc)

Train accuracy: 0.639275
Validation accuracy: 0.5314


In [ ]:
def predict_sentiment(review):

    encoded = encode(review, vocab)
    encoded = pad_sequence(encoded, MAX_LEN)

    x = torch.tensor(
        encoded,
        dtype=torch.long
    ).unsqueeze(0)

    x = x.to(device)
    
    model.eval()

    with torch.no_grad():
        logit = model(x)
        probability = torch.sigmoid(logit).item()

    if probability >= 0.5:
        return "Positive", probability
    else:
        return "Negative", probability

In [ ]:
print(predict_sentiment("This movie was absolutely amazing"))
print(predict_sentiment("This was a terrible experience"))
print(predict_sentiment("I really enjoyed this product"))
print(predict_sentiment("Complete waste of money"))

('Positive', 0.5576763153076172)
('Positive', 0.5576763153076172)
('Positive', 0.5140368342399597)
('Positive', 0.5820891857147217)


In [ ]:
print("Train:")
print(train_df["label_num"].value_counts())

print("\nValidation:")
print(val_df["label_num"].value_counts())

print("\nTest:")
print(test_df["label_num"].value_counts())

Train:
label_num
1    20000
0    20000
Name: count, dtype: int64

Validation:
label_num
0    2500
1    2500
Name: count, dtype: int64

Test:
label_num
0    2500
1    2500
Name: count, dtype: int64


In [ ]:
for i in range(10):
    print(
        train_df.iloc[i]["label_num"],
        "→",
        train_df.iloc[i]["review"][:150]
    )

1 → Although this movie has some weaknesses, it is worth seeing. I chose it because of the cast, and applaud Bonham Carter and Branagh for choosing roles 
0 → I hate to comment on something I didn't finish, but if I spare one person what I sat through for almost an hour before turning it off in disgust, it w
0 → The movie is steeped in religion, so it is impossible to separate it from religion in commenting upon it. In my opinion, this movie pretends to explor
0 → I picked out this DVD out of the cheepo bin at Walmart because the cover showed one of the planes I flew during Viet-Nam (C-123k). I did not fly for A
0 → Jack Brooks (Trevor Matthews) is a college student with some severe anger issues. His family was brutally murdered when he was a child by a monster, a
1 → Cunningly interesting Western from a director who had few peers in the genre. Much like other Anthony Mann pictures, The Far Country blends a potent p
1 → This story of the troubles caused by an over-possessive, overpower

In [ ]:
model.eval()

positive = 0
negative = 0

with torch.no_grad():
    for x, y in val_loader:

        x = x.to(device)
        y = y.to(device)
        
        logits = model(x)

        predictions = (torch.sigmoid(logits) >= 0.5)

        positive += predictions.sum().item()
        negative += (~predictions).sum().item()

print("Positive predictions:", positive)
print("Negative predictions:", negative)

Positive predictions: 2495
Negative predictions: 2505


In [ ]:
print("Validation actual labels:")
print(val_df["label_num"].value_counts())

Validation actual labels:
label_num
0    2500
1    2500
Name: count, dtype: int64


In [ ]:
test_loss, test_accuracy = evaluate(
    model,
    test_loader,
    criterion
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

Test Loss: 0.8014491830661798
Test Accuracy: 0.528
